## Santander Customer Transaction Prediction

This project predicts whether a bank customer will make a specific transaction in the future, using 200 anonymized numeric features. It is a binary classification problem with a strong class imbalance (about 10% positives), scored by AUC.

## Approach
1. Load the data and explore the target balance and features
2. Train a gradient boosting baseline and evaluate AUC on a validation split
3. Generate probabilities and create the submission file
4. Submit to Kaggle and record the score

In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c santander-customer-transaction-prediction
!unzip -oq santander-customer-transaction-prediction.zip

Paste your Kaggle API token and press Enter: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.8 MB/s eta 0:00:00
100% 250M/250M [00:02<00:00, 98.8MB/s]



In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
train.shape, test.shape

((200000, 202), (200000, 201))

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Columns: 202 entries, ID_code to var_199
dtypes: float64(200), int64(1), object(1)
memory usage: 308.2+ MB


In [5]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Columns: 201 entries, ID_code to var_199
dtypes: float64(200), object(1)
memory usage: 306.7+ MB


In [6]:
train.sample(1)

,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
169633,train_169633,0,12.7897,-0.642,11.8096,7.9308,8.6535,-15.7751,5.3142,19.9877,...,6.3016,10.9418,2.3307,4.923,18.2488,-2.1417,2.0992,8.4679,16.4752,-9.7934


In [7]:
test.sample(1)

,ID_code,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
20175,test_20175,15.0688,6.2822,8.3717,7.6953,13.6671,-18.3612,5.7681,12.2904,2.4884,...,4.7171,8.984,4.534,3.5329,18.52,-1.1854,-1.8211,9.5839,16.4811,12.9059


In [8]:
train["target"].value_counts(normalize=True)

,proportion
target,
0,0.89951
1,0.10049


In [9]:
train.columns[:5].tolist()

['ID_code', 'target', 'var_0', 'var_1', 'var_2']

In [10]:
test_ids = test["ID_code"]

x = train.drop(columns=["ID_code", "target"])
y = train["target"]

In [11]:
x.shape, y.shape

((200000, 200), (200000,))

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=42)
model.fit(x_train, y_train)

proba = model.predict_proba(x_val)[:, 1]
print("Validation AUC:", round(roc_auc_score(y_val, proba), 4))

Validation AUC: 0.8788


In [13]:
model.fit(x, y)

x_test = test.drop(columns=["ID_code"])
proba_test = model.predict_proba(x_test)[:, 1]

pd.DataFrame({"ID_code": test_ids, "target": proba_test}).to_csv("submission.csv", index=False)

In [14]:
!kaggle competitions submit -c santander-customer-transaction-prediction -f submission.csv -m "HistGradientBoosting baseline"

100% 6.02M/6.02M [00:00<00:00, 8.19MB/s]
99 submissions remaining today.
Successfully submitted to Santander Customer Transaction Prediction

In [15]:
meta = {
    "features": list(x.columns),
    "mean": x.mean().to_dict(),
    "std": x.std().to_dict(),
    "base_rate": float(y.mean()),
}

import pickle, os
pickle.dump(model, open("santander_model.pkl", "wb"))
pickle.dump(meta, open("santander_meta.pkl", "wb"))
print("model MB:", round(os.path.getsize("santander_model.pkl") / 1e6, 2))

model MB: 1.49
